# Course: 65007 - NLP and Speech Analysis
 **Program:** Intelligent Systems  
 **Course coordinator:** Dr. Sharon Yalov-Handzel

**Submission for:**  Assignment 2   
**by**  
  - Michael Berger, 318063864  
  - Barack Samuni, 318299625

# 1. Word2Vec

## a. Write Python program to implement Skip-gram Word2Vec algorithm.
Barak

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.tokenize import word_tokenize

# Download necessary NLTK resources
nltk.download('punkt')


class SkipGramDataset(Dataset):
    def __init__(self, tokenized_corpus, word_to_index, window_size):
        """
        Custom dataset for Skip-gram Word2Vec.
        Generates target-context word pairs based on the pre-tokenized corpus.
        """
        self.pairs = []  # Store target-context pairs

        # Generate target-context pairs
        for sentence in tokenized_corpus:
            sentence_indices = [word_to_index[word] for word in sentence]
            for i, target_index in enumerate(sentence_indices):
                # Create a context window around the target word
                context_indices = sentence_indices[max(0, i - window_size):i] + \
                                  sentence_indices[i + 1:min(len(sentence_indices), i + window_size + 1)]
                for context_index in context_indices:
                    # Append the (target, context) pair to the list
                    self.pairs.append((target_index, context_index))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        return self.pairs[index]


class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_size):
        """
        Skip-gram Word2Vec model with trainable embeddings.
        """
        super(SkipGramModel, self).__init__()
        self.input_embeddings = nn.Embedding(vocab_size, embedding_size)    # target words to dense embedding vectors
        self.output_embeddings = nn.Embedding(vocab_size, embedding_size)   # context words to dense embedding vectors

    def forward(self, target_words, context_words):
        """
        Forward pass to compute the logits (scores) for target-context word pairs.
        """
        # Embedding lookup for target and context words
        target_embeds = self.input_embeddings(target_words)     # Shape: (batch_size, embedding_size)
        context_embeds = self.output_embeddings(context_words)  # Shape: (batch_size, embedding_size)

        # Compute dot product (logits) between target and context embeddings
        logits = torch.sum(target_embeds * context_embeds, dim=1)  # Shape: (batch_size)

        return logits

def train_skipgram_model(corpus, embedding_size=10, window_size=2, learning_rate=0.01, epochs=10, batch_size=64):
    """
    Trains a Skip-gram Word2Vec model with PyTorch. Handles tokenization and vocab generation internally.
    """
    # Tokenization and Vocabulary Creation
    tokenized_corpus = [word_tokenize(sentence.lower()) for sentence in corpus]
    words = [word for sentence in tokenized_corpus for word in sentence]
    vocab = list(set(words))
    word_to_index = {word: i for i, word in enumerate(vocab)}
    index_to_word = {i: word for word, i in word_to_index.items()}
    vocab_size = len(word_to_index)

    # Create Skip-gram dataset and dataloader
    dataset = SkipGramDataset(tokenized_corpus, word_to_index, window_size)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Initialize the SkipGram model
    model = SkipGramModel(vocab_size, embedding_size)
    criterion = nn.CrossEntropyLoss()  # Negative log likelihood loss with softmax
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Get the training device (CPU or GPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")
    model.to(device)

    # Training loop
    for epoch in range(epochs):
        total_loss = 0
        for target, context in dataloader:
            # Send tensors to the training device
            target = target.to(device)
            context = context.to(device)

            # Forward pass
            logits = model.input_embeddings(target) @ model.output_embeddings.weight.T  # Shape: (batch_size, vocab_size)

            # Compute loss
            loss = criterion(logits, context)

            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss:.4f}")

    # Extract word embeddings to a dictionary
    embeddings = model.input_embeddings.weight.cpu().detach().numpy()
    embedding_dict = {index_to_word[i]: embeddings[i] for i in range(vocab_size)}

    return embedding_dict

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## b. Write Python program that implements CBOW Word2Vec algorithm
Michael

## c. Apply both programs to the following text:

```
i. The bank is located near the river.
ii. The bank approved my loan application.
iii. He rose from his chair to close the window.
iv. The rose bloomed beautifully in the garden.
v. The lead actor delivered a stunning performance.
vi. Exposure to lead is harmful to health.
vii. She is reading a book in the library.
viii. The book mentioned a fascinating historical event.
ix. I need to file a report for my manager.
x. He lost the file containing important documents.
```

Skip-gram

In [2]:
corpus = [
    "The bank is located near the river.",
    "The bank approved my loan application.",
    "He rose from his chair to close the window.",
    "The rose bloomed beautifully in the garden.",
    "The lead actor delivered a stunning performance.",
    "Exposure to lead is harmful to health.",
    "She is reading a book in the library.",
    "The book mentioned a fascinating historical event.",
    "I need to file a report for my manager.",
    "He lost the file containing important documents."
]
embedding_dict_skipgram = train_skipgram_model(corpus)
embedding_dict_skipgram

Training device: GPU (CUDA)
Epoch 1/10, Loss: 38.3173
Epoch 2/10, Loss: 36.2217
Epoch 3/10, Loss: 33.2955
Epoch 4/10, Loss: 31.9203
Epoch 5/10, Loss: 30.2215
Epoch 6/10, Loss: 29.2343
Epoch 7/10, Loss: 27.2867
Epoch 8/10, Loss: 27.0708
Epoch 9/10, Loss: 25.6874
Epoch 10/10, Loss: 24.8552


{'lead': array([-1.2440723 , -0.40988415, -0.42608365,  1.7652538 , -0.24989493,
         1.5518929 ,  1.497566  ,  0.85501844, -0.2212901 ,  0.4096871 ],
       dtype=float32),
 'chair': array([ 0.6801907 ,  0.85449713,  0.3242614 ,  0.24592015,  0.43795255,
         1.0728728 ,  0.5301901 ,  0.21542634,  0.61838883, -0.44913775],
       dtype=float32),
 'the': array([ 0.55791765,  0.07948576,  1.8612902 , -0.6079762 , -0.80913377,
         0.00218629,  1.0339825 , -1.3844576 ,  0.6283675 , -0.14156018],
       dtype=float32),
 'in': array([ 0.66997397, -1.1697389 ,  0.9088643 , -0.02965271,  0.6554351 ,
         0.88757175,  0.32147583,  0.20041132, -0.44513234, -0.2892619 ],
       dtype=float32),
 'exposure': array([ 0.31027466, -0.28476334, -0.02758825,  0.38080463, -0.17919281,
         0.3466661 , -0.12261394, -0.7275818 , -0.625998  , -2.4661078 ],
       dtype=float32),
 'harmful': array([ 0.65314835, -0.42260212,  0.19332421,  0.28587472,  1.3593158 ,
         1.8026093 ,  0.

CBOW

## d. What is the difference between the embeddings? 
Explain the results.

## e. Can you find a text that 
Its embedding will be similar in these two algorithms?

## f. Repeat step c with different window sizes. 
Is there a significant change?

### Skip-gram

In [3]:
import pandas as pd  # Import pandas to handle the DataFrame

window_sizes = [1, 2, 3, 4, 5]

# To hold embeddings for comparison and results
previous_embedding_dict = None
differences_data = []  # To store differences

for window_size in window_sizes:
    embedding_dict = train_skipgram_model(corpus, window_size=window_size)
    print(f"Window size: {window_size}\n")
    print("----------------------------------------\n")

    if previous_embedding_dict is not None:
        row_data = {}  # Row data for the DataFrame

        for word in embedding_dict:
            if word in previous_embedding_dict:
                difference = embedding_dict[word] - previous_embedding_dict[word]
                row_data[word] = difference  # Store difference
            else:
                print(f"Word: {word} is new in the current embedding.\n")

        differences_data.append(row_data)  # Append the differences for this window size comparison
        print("----------------------------------------\n")

    previous_embedding_dict = embedding_dict  # Update for the next comparison

# Create a DataFrame from the differences data
comparison_labels = [f"{window_sizes[i]}-{window_sizes[i+1]}" for i in range(len(window_sizes)-1)]
differences_df = pd.DataFrame(differences_data, index=comparison_labels)
print("Differences DataFrame:")
differences_df

Training device: GPU (CUDA)
Epoch 1/10, Loss: 22.8439
Epoch 2/10, Loss: 21.1828
Epoch 3/10, Loss: 19.5191
Epoch 4/10, Loss: 18.9438
Epoch 5/10, Loss: 17.7709
Epoch 6/10, Loss: 17.7107
Epoch 7/10, Loss: 17.4402
Epoch 8/10, Loss: 16.5747
Epoch 9/10, Loss: 15.6783
Epoch 10/10, Loss: 15.3562
Window size: 1

----------------------------------------

Training device: GPU (CUDA)
Epoch 1/10, Loss: 39.5401
Epoch 2/10, Loss: 36.1717
Epoch 3/10, Loss: 34.3387
Epoch 4/10, Loss: 32.7906
Epoch 5/10, Loss: 31.0127
Epoch 6/10, Loss: 29.6177
Epoch 7/10, Loss: 29.3725
Epoch 8/10, Loss: 26.6430
Epoch 9/10, Loss: 26.5741
Epoch 10/10, Loss: 25.2050
Window size: 2

----------------------------------------

----------------------------------------

Training device: GPU (CUDA)
Epoch 1/10, Loss: 44.3841
Epoch 2/10, Loss: 41.1846
Epoch 3/10, Loss: 38.4979
Epoch 4/10, Loss: 36.2214
Epoch 5/10, Loss: 34.2438
Epoch 6/10, Loss: 32.5939
Epoch 7/10, Loss: 31.1595
Epoch 8/10, Loss: 29.8900
Epoch 9/10, Loss: 28.8100
Ep

,lead,chair,the,in,exposure,harmful,library,application,lost,near,...,my,delivered,for,is,river,stunning,health,need,actor,to
1-2,"[-1.2806672, -0.8138023, 0.41850752, -2.735672...","[-0.89775753, 0.52893114, 0.7132163, -0.831357...","[-1.3603925, -0.41665158, -1.1366961, -0.11122...","[-0.76865625, -1.9265984, -0.42184743, -2.1476...","[3.179974, -1.2142799, 4.1058636, 1.5871679, 0...","[-0.39681888, 1.1439295, 3.3662515, -1.1235759...","[0.9016199, 0.30254513, -0.25291103, -0.821122...","[1.5579313, 0.27924037, -0.79256546, -0.176348...","[-2.6369634, -2.4555535, -1.1080563, -0.876430...","[-1.0617769, -0.03219211, -0.3435216, -0.86963...",...,"[-1.1161695, 1.5282903, 0.8793084, -3.5630221,...","[0.44889218, -0.25732255, -1.7565702, -1.00628...","[0.6860809, -0.81409806, -1.3549134, -1.166009...","[1.9430239, -1.5437881, 0.66338736, 0.8011321,...","[0.86543024, 1.3161805, 0.2085433, -2.591698, ...","[-1.9068038, -1.1344316, 0.3947307, -0.2048859...","[1.1721637, 0.7897935, -1.3665811, -1.3529668,...","[1.1705083, -2.146698, 0.9372696, 0.117791295,...","[-1.5631822, -0.9056266, 1.2333416, 0.51546973...","[-0.78269327, -0.1429525, 2.4802809, -0.509379..."
2-3,"[1.4537423, 0.7056808, 0.15186906, 0.30154854,...","[0.074309856, 0.729681, 0.5480814, 0.104981355...","[1.1838496, 1.8101461, 1.1387392, 1.5074103, -...","[2.2931197, -0.016052932, -0.40244037, 0.49519...","[-2.267091, 0.4380796, -1.1156125, -2.360386, ...","[-0.2081624, -2.2823482, -0.9447591, 0.0718858...","[-0.9986316, -0.08838403, 0.9376968, 0.0620961...","[0.99774617, -0.38597876, -0.12722936, 0.92432...","[-0.12156014, -0.8196596, 0.30671972, 0.524164...","[1.2939788, -1.183372, 0.89025867, 0.018116206...",...,"[-0.411622, -2.6978116, 0.8691261, 3.6181765, ...","[-1.8682382, -1.7957065, -0.5518913, 1.129222,...","[0.3493051, -0.04070288, 0.6018811, -0.3455990...","[0.17875275, -0.54194355, 0.17027378, 0.664281...","[-0.7770081, -1.755578, -1.6318849, 1.4435313,...","[0.108977675, 0.28702033, 0.3827917, 1.6689912...","[0.06831357, -0.5382347, 0.27933693, 2.492631,...","[-1.2964361, 1.5376115, -0.2986867, -2.027732,...","[0.35640526, 1.6461635, -1.4878558, -0.8321234...","[-0.4253381, 0.5390613, -1.2024932, 0.31946874..."
3-4,"[-2.2987628, -1.7064602, -1.2731407, 1.042717,...","[0.49368417, -1.3650936, -1.0588331, -0.030805...","[-1.282049, -1.0791821, -0.4965928, -0.3829781...","[-0.52570814, -0.27177417, -0.17450565, -0.381...","[0.25578052, 1.7629409, 0.7052779, -0.00225734...","[-0.21618623, 0.2674924, -0.7239182, -1.774094...","[-0.21037392, -0.24545093, -1.713299, 0.745301...","[0.8047691, -1.1238124, -0.9487227, -0.7874166...","[-0.31432307, 1.0785285, -0.42506492, -0.70976...","[-0.22781736, 2.027943, -0.57665193, -1.374528...",...,"[-1.0406065, 0.27470288, -1.5073563, -2.462646...","[1.0003157, -0.3815328, 1.5246041, -3.1036515,...","[-1.0426561, 0.4940427, -1.2398812, 0.10202443...","[-0.9611381, 3.359333, -0.514774, -2.6245947, ...","[0.8227258, 2.7344167, 0.52710855, 0.35236418,...","[1.3605413, 0.7293855, 0.45472908, -0.5615702,...","[-1.5965447, -0.7868206, 1.917283, -1.1882108,...","[-0.7949152, -0.42842332, -1.0698677, 0.139891...","[1.6404605, -0.66863275, -0.7476412, -0.175145...","[0.4108974, -1.4382293, 0.069178484, 1.2496957..."
4-5,"[2.341978, 0.2652824, 0.89030653, -0.8270793, ...","[-0.47037563, -0.27200747, -1.1270841, -0.4110...","[1.2086724, -0.8393943, 1.4567415, 1.3379607, ...","[-1.2666504, 0.7849161, 1.5183103, 0.6625756, ...","[0.16629678, -1.8220706, -0.38851708, 1.569186...","[0.8343025, 0.7970239, -0.06382467, 1.0720812,...","[-0.22773054, -0.92805696, 0.6764782, -0.37204...","[0.11386913, -1.1126025, 1.4609818, 1.4813869,...","[-0.5226497, 0.6916228, 1.8516283, -0.02772800...","[-0.41439778, -0.5556717, 0.60204506, 0.745511...",...,"[1.0351865, 0.12187207, -1.2439424, 2.2172904,...","[-0.91526484, 1.5043472, -0.069674514, 1.18561...","[0.5408225, 0.30084908, 1.7517827, 0.16931033,...","[0.8430995, -2.4502988, -0.3097219, 1.0011846

We can see that there differences between different window sizes. However, it doesn't seem to have a constant trend. Increasing the window size is supposed to make the model be better at recognizing context, and thus the weights should change drastically for double-meaning words (such as bank) and we can see that it does.

### CBOW

## g. Compare these two models 
In terms of capturing the syntactic and the semantic relationship between words.

## h. Demonstrate the difference between CBOW and Skip-grams
In terms of cosine similarity between the following words:
 - bank, rose, lead, book and file.

### Skip-gram

In [4]:
import pandas as pd
from scipy.spatial.distance import cosine

selected_words = ['bank', 'rose', 'lead', 'book', 'file']  # Words to calculate cosine similarity for

# Initialize an empty DataFrame to store cosine similarities
cosine_similarity_df = pd.DataFrame(index=selected_words, columns=selected_words)

# Calculate cosine similarity for selected word pairs
for word1 in selected_words:
    for word2 in selected_words:
        if word1 != word2:
            # Calculate cosine similarity (1 - cosine distance)
            similarity = 1 - cosine(embedding_dict_skipgram[word1], embedding_dict_skipgram[word2])
            cosine_similarity_df.at[word1, word2] = similarity
        else:
            # Similarity with itself is 1
            cosine_similarity_df.at[word1, word2] = 1.0

# Convert to float type (optional)
cosine_similarity_df = cosine_similarity_df.astype(float)

# Display the resulting DataFrame
cosine_similarity_df

,bank,rose,lead,book,file
bank,1.000000,0.114079,0.544377,0.465391,-0.073770
rose,0.114079,1.000000,-0.417166,0.239949,0.318385
lead,0.544377,-0.417166,1.000000,0.142842,-0.337738
book,0.465391,0.239949,0.142842,1.000000,0.124295
file,-0.073770,0.318385,-0.337738,0.124295,1.000000


### CBOW

## i. How can the subword embeddings be applied?
Michael

# 2. Create example sentences demonstrating: 
how contextual embeddings handle words with multiple meanings (polysemy) differently than static embeddings like Word2Vec.
Barak

In [6]:
import tensorflow_hub as hub
import tensorflow as tf
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

# Make sure to download NLTK data (if not already downloaded)
nltk.download('punkt')

# Load ELMo model from TensorFlow Hub
elmo = hub.load("https://tfhub.dev/google/elmo/3")

# Example sentences for demonstrating polysemy
example_sentences = [
    "He went to the bank to deposit some money.",  # bank as a financial institution
    "The fisherman docked his boat near the river bank.",  # bank as a riverbank
    "She rose from her chair to make an announcement.",  # rose as an action/past tense of rise
    "The rose smelled fragrant and bloomed beautifully.",  # rose as a flower
    "The project manager will lead the team during the meeting.",  # lead as to guide
    "Exposure to lead in paint can cause health issues."  # lead as a toxic substance
]

words_to_compare = ["bank", "rose", "lead"]  # Words to compare

# Function to get contextual embeddings using ELMo
def get_elmo_embeddings(sentences, word):
    """
    Extract contextual embeddings for a specific word from ELMo.
    Arguments:
    - sentences: List of sentences to process.
    - word: The target word to extract embeddings for.

    Returns:
    - A dictionary where keys are sentences and values are embeddings of the target word.
    """
    embeddings = {}

    for sentence in sentences:
        elmo_embeddings = elmo.signatures["default"](tf.constant([sentence]))["elmo"]  # Shape: (1, sentence_length, 1024)
        elmo_embeddings_np = elmo_embeddings.numpy().squeeze()  # Shape: (sentence_length, 1024)
        tokens = word_tokenize(sentence)  # Tokenize using nltk
        if word in tokens:
            word_index = tokens.index(word)  # Find the index of the word
            embeddings[sentence] = elmo_embeddings_np[word_index]  # Extract the embedding for the word
        else:
            embeddings[sentence] = None  # Word not found in the sentence
    return embeddings

# Get static embeddings from the Skip-gram Word2Vec model
def get_skipgram_embedding(word, embedding_dict_skipgram):
    """
    Extract the static embedding for a word from the Skip-gram Word2Vec model.
    """
    return embedding_dict_skipgram.get(word)

# Compare Skip-gram and ELMo embeddings
for word in words_to_compare:
    print(f"\nWord: '{word}'")

    # Contextual embeddings with ELMo
    print("Contextual Embeddings (varies by sentence):")
    elmo_embeddings = get_elmo_embeddings(example_sentences, word)
    for sentence, embedding in elmo_embeddings.items():
        if embedding is not None:
            print(f"  Sentence: {sentence}")
            print(f"    Embedding: {embedding[:5]}...")  # Print only the first 5 dimensions for brevity
        else:
            print(f"  Sentence: {sentence}")
            print("    Embedding: Word not found in the sentence.")

    # Static embedding using Skip-gram Word2Vec
    print("Static Embedding (same for all sentences):")
    skipgram_embedding = get_skipgram_embedding(word, embedding_dict_skipgram)
    if skipgram_embedding is not None:
        print(f"  Embedding: {skipgram_embedding[:5]}...")  # Print only the first 5 dimensions for brevity
    else:
        print("  Embedding: Word not found in the Word2Vec vocabulary.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Word: 'bank'
Contextual Embeddings (varies by sentence):
  Sentence: He went to the bank to deposit some money.
    Embedding: [-0.40484837  0.18932024  0.08644994  0.2684955   0.35230607]...
  Sentence: The fisherman docked his boat near the river bank.
    Embedding: [ 0.10449297  0.3106082  -0.563089   -0.53706545 -0.81824535]...
  Sentence: She rose from her chair to make an announcement.
    Embedding: Word not found in the sentence.
  Sentence: The rose smelled fragrant and bloomed beautifully.
    Embedding: Word not found in the sentence.
  Sentence: The project manager will lead the team during the meeting.
    Embedding: Word not found in the sentence.
  Sentence: Exposure to lead in paint can cause health issues.
    Embedding: Word not found in the sentence.
Static Embedding (same for all sentences):
  Embedding: [ 0.08255051 -0.5002217   0.80496657  0.10599652  0.94017744]...

Word: 'rose'
Contextual Embeddings (varies by sentence):
  Sentence: He went to the bank to depo

# 3. Propose metrics
For evaluating word embeddings that can differentiate between syntactic and semantic relationships.
Michael

# 4. Use the Gensim library 
To train a Word2Vec model on a custom corpus.  
Barak

In [7]:
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec
from nltk.tokenize import word_tokenize, sent_tokenize
import nltk

# Ensure NLTK dependencies are downloaded
nltk.download('punkt')

class EpochLogger(CallbackAny2Vec):
    """Callback to log information about training progress."""
    def __init__(self):
        self.epoch = 0

    def on_epoch_begin(self, model):
        print(f"Epoch {self.epoch} starting...")

    def on_epoch_end(self, model):
        print(f"Epoch {self.epoch} finished.")
        self.epoch += 1

def train_word2vec(corpus, size=100, window=5, min_count=1, sg=1, epochs=10, workers=4):
    """
    Generalized function to train a Word2Vec model using Gensim, with sentence tokenization
    using NLTK's tokenizer.

    Args:
        corpus (list of str): A list of sentences to train the model on.
        size (int): Dimensionality of the Word2Vec embeddings. Default is 100.
        window (int): Maximum distance between the current and predicted word within a sentence. Default is 5.
        min_count (int): Ignore words with total frequency lower than this. Default is 1.
        sg (int): 1 for skip-gram; 0 for CBOW. Default is 1 (skip-gram).
        epochs (int): Number of training epochs. Default is 10.
        workers (int): Number of threads to use during model training. Default is 4.

    Returns:
        Word2Vec: Trained Word2Vec model.
    """
    # Tokenize each sentence using NLTK
    tokenized_corpus = [word_tokenize(sentence.lower()) for sentence in corpus]

    # Instantiate the logger for monitoring progress
    epoch_logger = EpochLogger()

    # Initialize and train the Word2Vec model
    model = Word2Vec(
        sentences=tokenized_corpus,
        vector_size=size,
        window=window,
        min_count=min_count,
        sg=sg,
        epochs=epochs,
        workers=workers,
        callbacks=[epoch_logger]
    )
    print("Model training completed.")
    return model

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\barak\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## a. Evaluate the quality of embeddings 
By calculating the cosine similarity for the following word pairs:
```
i. "king" and "queen"  
ii. "man" and "woman"
iii. "apple" and "orange"
```

In [8]:

# Define a logical sentence containing the specified words
corpus = ["The king and the queen ruled the land, while the man and the woman tended to the apple orchard near the orange grove."]

# Train a Word2Vec model on the corpus
model = train_word2vec(corpus=corpus, size=100, window=5, min_count=1, sg=1, epochs=10, workers=4)

# Extract embeddings for the specified words
embedding_dict = {word: model.wv[word] for word in ["king", "queen", "man", "woman", "apple", "orange"]}

# Define the words for cosine similarity calculation
selected_words = ["king", "queen", "man", "woman", "apple", "orange"]

# Initialize a DataFrame to store cosine similarities
cosine_similarity_df = pd.DataFrame(index=selected_words, columns=selected_words)

# Calculate cosine similarity for each word pair
from scipy.spatial.distance import cosine

for word1 in selected_words:
    for word2 in selected_words:
        if word1 != word2:
            # Calculate cosine similarity
            similarity = 1 - cosine(embedding_dict[word1], embedding_dict[word2])
            cosine_similarity_df.at[word1, word2] = similarity
        else:
            # Set self-similarity to 1
            cosine_similarity_df.at[word1, word2] = 1.0

# Convert to float type (optional for better display purposes)
cosine_similarity_df = cosine_similarity_df.astype(float)

# Display the resulting DataFrame of cosine similarities
cosine_similarity_df

Epoch 0 starting...
Epoch 0 finished.
Epoch 1 starting...
Epoch 1 finished.
Epoch 2 starting...
Epoch 2 finished.
Epoch 3 starting...
Epoch 3 finished.
Epoch 4 starting...
Epoch 4 finished.
Epoch 5 starting...
Epoch 5 finished.
Epoch 6 starting...
Epoch 6 finished.
Epoch 7 starting...
Epoch 7 finished.
Epoch 8 starting...
Epoch 8 finished.
Epoch 9 starting...
Epoch 9 finished.
Model training completed.


,king,queen,man,woman,apple,orange
king,1.000000,-0.044282,0.041448,0.015665,0.007389,-0.260190
queen,-0.044282,1.000000,-0.028988,-0.255465,0.253951,0.109047
man,0.041448,-0.028988,1.000000,0.096001,0.116301,-0.148974
woman,0.015665,-0.255465,0.096001,1.000000,0.004475,-0.010320
apple,0.007389,0.253951,0.116301,0.004475,1.000000,-0.031320
orange,-0.260190,0.109047,-0.148974,-0.010320,-0.031320,1.000000


## b. Write a brief explanation of the results.

We can see that the word couples: "king" and "queen", "man" and "woman", "apple" and "orange" may have a similar context, but have a pretty weak cosine similarity when it comes to a Word2vec model. It could mean that a Word2Vec model looks at the words independently and thus words with similar context might not necessarily have a strong cosine similarity, or it could be that the embedding captures another similarity (perhaps queen and woman might lead to a stronger cosine similarity for instance).

# 5. Train the GloVe model 
Using the glove-python package on a subset of a publicly available dataset (e.g., Wikipedia, or a smaller custom corpus).

Michael

## a. Use t-SNE or PCA 
To visualize the embeddings in 2D.

## b. Analyze the clustering patterns observed in the visualization.